# AutoResearch FetchSlide — Colab T4 GPU

Run the from-scratch FetchSlide RL loop on a Colab T4 GPU. Same code as the local `autoresearch` package; only the device changes (`cuda`).

1. Run all cells in order.
2. Cell 2 clones the repo.
3. Cell 3 runs the reference-recipe training (pure sparse + TD3 + reference cadence) on T4.
4. Cell 4 runs the multi-agent autoresearch loop.

All checkpoints and summaries are written under `AUTORESEARCH_RUNS` (default `/content/autoresearch-runs`).

In [1]:
!pip -q install gymnasium gymnasium-robotics torch 2>&1 | tail -3

In [2]:
import torch
assert torch.cuda.is_available(), 'CUDA GPU required: Runtime > Change runtime type > T4 GPU'
print('CUDA device:', torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0) or torch.cuda.get_device_capability(0)[0] >= 7, 'Expected a T4-class CUDA GPU'


CUDA device: Tesla T4


In [3]:
import os, sys, shutil, subprocess

REPO = '/content/autoresearch'
RUNS = os.environ.get('AUTORESEARCH_RUNS', '/content/autoresearch-runs')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/DeconvFFT/fetch-and-slide-HRE-PRE.git', REPO], check=True)
os.makedirs(RUNS, exist_ok=True)
os.environ['AUTORESEARCH_RUNS'] = RUNS
sys.path.insert(0, REPO)
print('REPO', REPO, 'RUNS', RUNS)

REPO /content/autoresearch RUNS /content/autoresearch-runs


## Reference-recipe training (pure sparse + exact DDPG + reference cadence)

This matches the reference HER+DDPG recipe that hit 80%: pure sparse reward, TD3 (better than DDPG per literature), and the reference cadence (rollouts_per_cycle=2, optimsteps=40). Runs on T4 GPU.

In [4]:
import json
cfg = {
    'algorithm': 'ddpg',
    'train_episodes': 20000, 'horizon': 50, 'eval_episodes': 50, 'batch_size': 256,
    'warmup_steps': 0, 'updates_per_step': 1, 'eval_every': 2000, 'log_every': 100,
    'device': 'cuda', 'her_future': 4, 'her_ratio': 0.8,
    'per': False, 'hper': False, 'dense_reward': False, 'success_bonus': 0.0,
    'reach_coef': 0.0, 'reach_contact_bonus': 0.0, 'push_coef': 0.0, 'goal_bonus': 0.0,
    'goal_bonus_radius': 0.4, 'actor_l2': 1.0, 'scripted_rollouts': 0, 'scripted_every': 0,
    'rollouts_per_cycle': 2, 'optimsteps': 40, 'replay_capacity': 1000000,
}
cfg_path = f'{RUNS}/ref_recipe.json'
os.makedirs(RUNS, exist_ok=True)
with open(cfg_path, 'w') as f:
    json.dump(cfg, f)
print('EXACT DDPG reference config written:', cfg_path)
print('total steps:', cfg['train_episodes'] * cfg['horizon'])


EXACT DDPG reference config written: /content/autoresearch-runs/ref_recipe.json
total steps: 1000000


In [ ]:
import subprocess, sys, os
cfg_path = f'{RUNS}/ref_recipe.json'
out = f'{RUNS}/ref_recipe'
log_path = f'{RUNS}/ref_recipe.log'
pid_path = f'{RUNS}/ref_recipe.pid'
env = dict(os.environ); env['PYTHONPATH'] = REPO + os.pathsep + env.get('PYTHONPATH', '')
# Detach training from this notebook cell. Colab can reconnect/poll without
# holding a foreground subprocess open for the entire multi-hour run.
log = open(log_path, 'a', buffering=1)
proc = subprocess.Popen([sys.executable, '-m', 'autoresearch.worker', '--config', cfg_path, '--output', out],
                       cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT,
                       start_new_session=True)
log.close()
open(pid_path, 'w').write(str(proc.pid))
print('training started in background; pid=', proc.pid)
print('log:', log_path)
print('rerun the monitor cell to inspect progress')


In [ ]:
from pathlib import Path
import json, os
log_path = Path(f'{RUNS}/ref_recipe.log')
pid_path = Path(f'{RUNS}/ref_recipe.pid')
if log_path.exists():
    print(log_path.read_text()[-6000:])
else:
    print('log not created yet')
if pid_path.exists():
    pid = int(pid_path.read_text())
    try:
        os.kill(pid, 0)
        print('status: RUNNING, pid=', pid)
    except ProcessLookupError:
        print('status: EXITED')
try:
    m = json.load(open(f'{RUNS}/ref_recipe/metrics.json'))
    print('FINAL score:', round(m['score'], 4), 'success:', m['metrics']['success_rate'], 'dist:', round(m['metrics']['mean_final_distance'], 4), 'contact:', m.get('contact_rate'))
except FileNotFoundError:
    print('final metrics not written yet')


## Multi-agent autoresearch loop

Runs the Karpathy-style autonomous loop with the two-model proposal system (pro strategist + flash implementor) on T4. Requires an OpenRouter API key.

In [ ]:
import os
os.environ['OPENROUTER_API_KEY'] = 'YOUR_OPENROUTER_KEY_HERE'  # <-- set your key
print('key set:', bool(os.environ['OPENROUTER_API_KEY'] and os.environ['OPENROUTER_API_KEY'] != 'YOUR_OPENROUTER_KEY_HERE'))

In [ ]:
import subprocess, sys, os
# Run multi-agent improvements from the exact DDPG baseline on CUDA.
os.environ['AUTORESEARCH_ALGORITHM'] = 'ddpg'
os.environ['AUTORESEARCH_TRAIN_EPISODES'] = '2000'
os.environ['AUTORESEARCH_MAX_EPISODES'] = '2000'
os.environ['AUTORESEARCH_EVAL_EPISODES'] = '10'
os.environ['AUTORESEARCH_BATCH_SIZE'] = '256'
os.environ['AUTORESEARCH_DEVICE'] = 'cuda'
os.environ['AUTORESEARCH_DENSE_REWARD'] = 'false'
os.environ['AUTORESEARCH_HER_FUTURE'] = '4'
os.environ['AUTORESEARCH_ROLLOUTS_PER_CYCLE'] = '2'
os.environ['AUTORESEARCH_OPTIMSTEPS'] = '40'
env = dict(os.environ); env['PYTHONPATH'] = REPO + os.pathsep + env.get('PYTHONPATH', '')
agent_log = f'{RUNS}/multi_agent.log'
agent_pid = f'{RUNS}/multi_agent.pid'
log = open(agent_log, 'a', buffering=1)
proc = subprocess.Popen([sys.executable, '-m', 'autoresearch.agent_loop', '--tag', 'colab-ddpg', '--iterations', '5'],
                       cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT,
                       start_new_session=True)
log.close()
open(agent_pid, 'w').write(str(proc.pid))
print('multi-agent loop started in background; pid=', proc.pid)
print('log:', agent_log)


In [ ]:
from pathlib import Path
import os, glob
log_path = Path(f'{RUNS}/multi_agent.log')
pid_path = Path(f'{RUNS}/multi_agent.pid')
if log_path.exists():
    print(log_path.read_text()[-6000:])
if pid_path.exists():
    pid = int(pid_path.read_text())
    try:
        os.kill(pid, 0)
        print('multi-agent status: RUNNING, pid=', pid)
    except ProcessLookupError:
        print('multi-agent status: EXITED')
best = sorted(glob.glob(f'{RUNS}/run-*/best_checkpoint.pt'))[-1] if glob.glob(f'{RUNS}/run-*/best_checkpoint.pt') else None
print('best checkpoint:', best)
print('results.tsv:')
!cd {REPO} && cat results.tsv 2>/dev/null | tail -20
